# Graphene/Ni(111) Interface: Registry, Separation and Work of Adhesion

## 0. Introduction

This notebook reproduces the structure and energetics of graphene on Ni(111) following the review:

> **Arjun Dahal, Matthias Batzill**
> "Graphene–nickel interfaces: a review"
> Nanoscale, 6(5), 2548. (2014)
> [DOI: 10.1039/c3nr05279f](https://doi.org/10.1039/c3nr05279f)

The review's structural facts (its section 2.1): graphene locks into a 1×1 registry on Ni(111);
LEED I–V and ion scattering identify the adsorbed structure as one carbon **atop** a first-layer Ni
and the other in the **fcc hollow**, 0.211 nm above the surface with a 0.005 nm buckling in which
the atop carbon sits further out. Its computed numbers come from
[Lahiri et al., New J. Phys. 13, 025001 (2011)](https://doi.org/10.1088/1367-2630/13/2/025001)
(open access), whose Table 1 is the quantitative target here:

| interface | work of adhesion (J/m²) | separation (Å) |
|---|---|---|
| fcc (atop + fcc hollow) | 0.81 | 2.16 |
| hcp (atop + hcp hollow) | 0.77 | 2.17 |
| hollow (fcc + hcp hollows) | 0.31 | 3.26 |

(The review's text quotes the hollow as 0.38 J/m²; the source paper's Table 1 says 0.31 — this
notebook targets the source.) The four candidate registries, in the review's own Fig. 1:

<img src="https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/optimization/optimization_interface_film_xy_position_graphene_nickel/0-figure-from-manuscript.webp" alt="The four registries of graphene on a close-packed metal surface" width="600"/>

The bridge registry (d) is not quantified in either paper — it is included here as an extra point
beyond the published set.

The published calculation (Lahiri et al., section 2.2) used **LDA, spin-polarized, with geometry
relaxation** — five Ni layers with the bottom two fixed — because "GGA does not provide an adequate
description of Ni–graphene bonding". This notebook follows that recipe in two tiers:

- **Fast (here, in minutes):** each registry relaxed with the
  [MACE-MP](https://github.com/ACEsuit/mace) machine-learned force field (+D3), with the bottom
  substrate layers fixed as in the paper; same-cell references give the work of adhesion. MACE is
  PBE-trained, and PBE is exactly the functional the paper rejects for this system — so its
  chemisorption values are expected to underbind, and the notebook prints them **against** the
  paper's rather than pretending. The structure side (registry, separation trend, buckling sign,
  the hollow's dispersion-bound minimum) is where the fast tier earns its keep.
- **Precise (platform jobs):** the paper's functional — **LDA** (pz, ultrasoft), spin-polarized,
  **with relaxation**, no dispersion correction (LDA binds this interface unaided, which is why the
  paper chose it) — for each registry plus the two same-cell references the work of adhesion needs.

**Prerequisite:** run
[optimization_interface_film_xy_position_graphene_nickel.ipynb](optimization_interface_film_xy_position_graphene_nickel.ipynb)
first — it creates and saves the base interface material this notebook loads.

## 1. Prepare the Environment
### 1.1. Install Packages


In [ ]:
from mat3ra.notebooks_utils.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.2. Set Parameters


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "./uploads"
BASE_MATERIAL_NAME = "Graphene_Nickel_interface"  # created by the companion structure notebook

# 4. MLFF parameters. MACE-MP-0 is trained on inorganic crystals and surfaces. The large model at
# float64 is not a preference: the medium model at float32 finds no chemisorbed minimum at all.
MACE_MODEL_FAMILY = "MACE-MP-0"
MACE_MODEL = "large"  # "small", "medium", "large"
MACE_DISPERSION = True  # D3; the hollow registry is dispersion-bound
MACE_DEFAULT_DTYPE = "float64"
MACE_DEVICE = "cpu"

# 5. Separation scan, in Angstrom — brackets the minima before relaxing. The window has to cover
# both published distances (2.16 A chemisorbed, 3.26 A for the hollow) with room on either side.
Z_SCAN_START = 1.8
Z_SCAN_STOP = 4.3
Z_SCAN_STEP = 0.25

# A chemisorbing registry has two minima: one where graphene bonds to the surface and one held
# only by dispersion, further out. Anything below 2.6 A is the chemisorbed branch by a wide
# margin either way (2.16 vs 3.26 A in the paper).
CHEMISORBED_BELOW = 2.6  # Angstrom

# 6. Relaxation — the paper's scheme: geometry optimization with the bottom substrate layers
# fixed. Relaxation is what produces the buckling, which is one of the published numbers.
FMAX = 0.02  # eV/A
FROZEN_SUBSTRATE_LAYERS = 2  # the paper fixes the bottom two of its five Ni layers

# 7. Workflow parameters
WORKFLOW_SEARCH_TERM = "total_energy.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Total Energy (Gr/Ni registry)"

# Method parameters — the published setup where the platform can express it. Lahiri et al. used
# LDA, spin-polarized, with relaxation, and no dispersion correction: LDA binds this interface
# unaided, and that is the stated reason they chose it over GGA.
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft; the platform carries the lda/pz set for Ni and C
FUNCTIONAL = "pz"  # LDA
MODEL_SUBTYPE = "lda"
ECUTWFC = 40   # GBRV publishes its ultrasoft set as a 40 / 200 Ry pair
ECUTRHO = 200

# K is at (1/3, 1/3), so in-plane divisions must be a multiple of three for the mesh to contain
# it, and a metal needs a dense mesh to resolve its Fermi surface.
SCF_KGRID = [12, 12, 1]

# Nickel is ferromagnetic — spin-polarized, started near its bulk moment (the paper's LDA value
# is 0.56 uB).
STARTING_MAGNETIZATION = {"Ni": 0.7}

# A spin-polarized metal slab is the hard case for SCF, and the platform defaults do not converge
# it: a first run stopped at "convergence NOT achieved after 100 iterations" with the total energy
# oscillating in its fourth decimal — charge sloshing, not divergence. Cold smearing, local-TF
# mixing and a smaller mixing fraction address exactly that.
SMEARING = "mv"  # Marzari-Vanderbilt cold smearing
DEGAUSS = 0.01  # Ry
ADDITIONAL_PARAMETERS = {
    "electrons": {
        "mixing_mode": "local-TF",
        "mixing_beta": 0.2,
        "electron_maxstep": 200,
    },
}

# 8. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 9. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30


## 2. Load the Base Interface

The base interface is created by the companion structure notebook and saved into `uploads/`.
It is required — this notebook does not substitute another material.


In [ ]:
from mat3ra.made.material import Material
from mat3ra.made.tools.modify import interface_get_part
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

base_interface = load_material_from_folder(FOLDER, BASE_MATERIAL_NAME)
if base_interface is None:
    raise RuntimeError(
        f"'{BASE_MATERIAL_NAME}' not found in {FOLDER} — run "
        "optimization_interface_film_xy_position_graphene_nickel.ipynb first."
    )

film_part = interface_get_part(base_interface, part=InterfacePartsEnum.FILM)
substrate_part = interface_get_part(base_interface, part=InterfacePartsEnum.SUBSTRATE)

_cart = base_interface.clone()
_cart.to_cartesian()
film_cart = film_part.clone(); film_cart.to_cartesian()
substrate_cart = substrate_part.clone(); substrate_cart.to_cartesian()

film_z = [c[2] for c in film_cart.basis.coordinates.values]
substrate_z = [c[2] for c in substrate_cart.basis.coordinates.values]
measured_gap = min(film_z) - max(substrate_z)

print(f"Material:  {base_interface.name}")
from collections import Counter
composition = dict(Counter(base_interface.basis.elements.values))
print(f"Composition: {composition}")
print(f"Atoms:     {len(base_interface.basis.elements.values)} "
      f"({len(film_cart.basis.elements.values)} film C, {len(substrate_cart.basis.elements.values)} substrate Ni)")
print(f"Film-substrate plane distance as built: {measured_gap:.3f} A")

visualize([{"material": base_interface, "title": base_interface.name}], repetitions=[3, 3, 1], rotation="-90x")

## 3. Place the Film at the High-Symmetry Registries

The registries are defined by where carbon atoms sit relative to the Ni(111) surface sites:
**top** (above a first-layer Ni), **hcp hollow** (above a second-layer Ni), **fcc hollow**
(above a third-layer Ni), and **bridge** (midpoint of two neighboring first-layer Ni).
The sites are measured from the structure itself — the top three Ni layers — and the film is
translated so one carbon sublattice lands on each site in turn.


In [ ]:
import numpy as np

cell_2d = np.array(_cart.lattice.vector_arrays)[:2, :2]

ni_xyz = np.array(substrate_cart.basis.coordinates.values)
z_values = sorted(set(np.round(ni_xyz[:, 2], 2)), reverse=True)
layer_tol = 0.5
layers = []
for z in z_values:
    if layers and abs(z - layers[-1][0]) < layer_tol:
        continue
    layers.append((z, ni_xyz[np.abs(ni_xyz[:, 2] - z) < layer_tol]))
if len(layers) < 3:
    raise RuntimeError(f"Need >= 3 Ni layers to locate the fcc and hcp sites, found {len(layers)}")

c_xyz = np.array(film_cart.basis.coordinates.values)
if len(c_xyz) != 2:
    raise RuntimeError(f"Expected a 1x1 graphene film (2 carbons), found {len(c_xyz)}")
c_a, c_b = c_xyz[0], c_xyz[1]

def nearest_image(site_xy, point_xy):
    """The periodic image of site_xy closest to point_xy."""
    images = [site_xy + i * cell_2d[0] + j * cell_2d[1] for i in (-1, 0, 1) for j in (-1, 0, 1)]
    return min(images, key=lambda s: np.linalg.norm(s - point_xy))

# Surface sites read off the structure itself: a first-layer Ni marks an atop site, a second-layer
# Ni projects onto the hcp hollow and a third-layer Ni onto the fcc hollow.
site_xy = {
    "atop": nearest_image(layers[0][1][0][:2], c_a[:2]),
    "hcp": nearest_image(layers[1][1][0][:2], c_a[:2]),
    "fcc": nearest_image(layers[2][1][0][:2], c_a[:2]),
}
def site_of(point_xy):
    """Which named site a carbon lands on. Refuses to guess when two are equidistant."""
    distances = {name: np.linalg.norm(nearest_image(site, point_xy) - point_xy)
                 for name, site in site_xy.items()}
    ordered = sorted(distances.items(), key=lambda kv: kv[1])
    if len(ordered) > 1 and abs(ordered[0][1] - ordered[1][1]) < 0.05:
        return None
    return ordered[0][0]

# The manuscript's Fig. 1: (a) hollow, (b) atop/fcc, (c) atop/hcp, (d) bridge. In a 1x1 cell the two
# carbon sublattices sit on two of the three named sites, which gives the first three. In the bridge
# registry neither carbon is on a site: the C-C bond straddles a first-layer Ni, which sits under the
# bond midpoint (Fig. 1d shows the vertical bonds running through the centres of the surface atoms).
bond_midpoint = (c_a[:2] + c_b[:2]) / 2
displacements = {"bridge": np.array([*(site_xy["atop"] - bond_midpoint), 0.0])}
for a_site in ("fcc", "atop", "hcp"):
    shift = np.array([*(site_xy[a_site] - c_a[:2]), 0.0])
    b_site = site_of(c_b[:2] + shift[:2])
    if b_site is None:
        raise RuntimeError(f"Carbon B is equidistant from two sites for the {a_site} placement")
    pair = {a_site, b_site}
    label = f"atop_{(pair - {'atop'}).pop()}" if "atop" in pair else "hollow"
    displacements[label] = shift

expected = {"hollow", "atop_fcc", "atop_hcp", "bridge"}
if set(displacements) != expected:
    raise RuntimeError(f"Registry derivation produced {set(displacements)}, expected {expected}")

bridge_offset = np.linalg.norm(nearest_image(site_xy["atop"], bond_midpoint + displacements["bridge"][:2])
                               - (bond_midpoint + displacements["bridge"][:2]))
if bridge_offset > 1e-6:
    raise RuntimeError(f"Bridge registry is off by {bridge_offset:.3f} A — no Ni under the bond midpoint")

print(f"{'registry':<12}{'manuscript Fig. 1':<22}{'film shift (A)'}")
for label, panel in (("hollow", "(a) hollow site"), ("atop_fcc", "(b) atop/'fcc' site"),
                     ("atop_hcp", "(c) atop/'hcp' site"), ("bridge", "(d) bridge site")):
    print(f"{label:<12}{panel:<22}{np.round(displacements[label][:2], 3)}")


In [ ]:
from mat3ra.made.tools.modify import interface_displace_part

def film_at(registry_label, plane_distance):
    displacement = displacements[registry_label] + np.array([0.0, 0.0, plane_distance - measured_gap])
    return interface_displace_part(base_interface, displacement=list(displacement))

preview = []
for label in displacements:
    m = film_at(label, measured_gap)
    m.name = f"{BASE_MATERIAL_NAME} {label}"
    preview.append({"material": m, "title": label})

visualize(preview, repetitions=[2, 2, 1])

## 4. Fast Tier: Relax Each Registry with MACE

Each registry is bracketed by a rigid scan, then **relaxed** — all atoms free, the bottom
substrate layers fixed, the paper's scheme — and the same-cell references (bare Ni slab,
free-standing graphene) are relaxed the same way, which turns total energies into a work of
adhesion: W = (E_slab + E_graphene − E_interface) / A. After each relaxation the registry is
re-measured from the final positions, so a structure that slid into a neighbouring registry
cannot be reported under the wrong name. Distances follow the paper's convention: the averaged
carbon height above the averaged top-Ni height; buckling is the height difference between the
two carbons, positive when the atop carbon sits further out.


In [ ]:
import importlib.util

from ase.constraints import FixAtoms
from ase.optimize import BFGS
from mat3ra.made.tools.convert import from_ase, to_ase
from mat3ra.notebooks_utils.mlff import create_mlff_calculator

# D3 needs the torch-dftd package. Where it is unavailable (the in-browser environment does not
# bundle it), MACE runs at plain PBE level — which is exactly the description the review rejects
# for this interface: chemisorption comes out unbound and the hollow registry loses its
# dispersion-bound minimum. The notebook states which picture it is computing.
dispersion_available = importlib.util.find_spec("torch_dftd") is not None
dispersion_active = MACE_DISPERSION and dispersion_available
if MACE_DISPERSION and not dispersion_available:
    print("torch-dftd is not available here: the fast tier runs WITHOUT dispersion — the")
    print("GGA-level picture the manuscript describes as inadequate for this interface.")

calculator = create_mlff_calculator(
    "mace",
    {
        "family": MACE_MODEL_FAMILY,
        "model": MACE_MODEL,
        "dispersion": dispersion_active,
        "default_dtype": MACE_DEFAULT_DTYPE,
        "device": MACE_DEVICE,
    },
)


In [ ]:
distances = np.arange(Z_SCAN_START, Z_SCAN_STOP + 1e-9, Z_SCAN_STEP)
n_carbon = len(film_cart.basis.elements.values)
film_elements = set(film_cart.basis.elements.values)
substrate_elements = set(substrate_cart.basis.elements.values)
EV_PER_A2_TO_J_PER_M2 = 16.0217663

def refine_minimum(x, y, i):
    if 0 < i < len(x) - 1:
        coefficients = np.polyfit(x[i - 1:i + 2], y[i - 1:i + 2], 2)
        d = float(-coefficients[1] / (2 * coefficients[0]))
        return d, float(np.polyval(coefficients, d))
    return float(x[i]), float(y[i])

def relax(atoms):
    """The paper's relaxation scheme: everything free except the bottom substrate layers."""
    symbols, z = atoms.get_chemical_symbols(), atoms.positions[:, 2]
    substrate_z = sorted({round(z[i], 1) for i, s in enumerate(symbols) if s in substrate_elements})
    held = [i for i, s in enumerate(symbols)
            if s in substrate_elements and round(z[i], 1) in substrate_z[:FROZEN_SUBSTRATE_LAYERS]]
    if held:
        atoms.set_constraint(FixAtoms(indices=held))
    atoms.calc = calculator
    BFGS(atoms).run(fmax=FMAX, steps=300)
    return atoms

def interface_geometry(atoms):
    """Distances per the paper's convention: averaged heights; buckling signed by the atop carbon."""
    symbols, pos = atoms.get_chemical_symbols(), atoms.positions
    carbon = [i for i, s in enumerate(symbols) if s in film_elements]
    nickel_z = [pos[i, 2] for i, s in enumerate(symbols) if s in substrate_elements]
    top_layer = [z for z in nickel_z if z > max(nickel_z) - 0.5]
    carbon_by_site = {site_of(pos[i, :2]): i for i in carbon}
    atop_index = carbon_by_site.get("atop")
    separation = float(np.mean([pos[i, 2] for i in carbon]) - np.mean(top_layer))
    if atop_index is not None:
        other = next(i for i in carbon if i != atop_index)
        buckling = float(pos[atop_index, 2] - pos[other, 2])
    else:
        buckling = float(abs(pos[carbon[0], 2] - pos[carbon[1], 2]))
    registry_now = frozenset(site_of(pos[i, :2]) for i in carbon)
    return separation, buckling, registry_now

# Same-cell references, relaxed under the same scheme
slab_atoms = relax(to_ase(substrate_part))
sheet_atoms = to_ase(film_part)
sheet_atoms.calc = calculator
BFGS(sheet_atoms).run(fmax=FMAX, steps=300)
E_slab, E_sheet = float(slab_atoms.get_potential_energy()), float(sheet_atoms.get_potential_energy())
cell = np.array(to_ase(base_interface).cell)
area = float(np.linalg.norm(np.cross(cell[0], cell[1])))

scan_results = {}
for label in displacements:
    energies = []
    for d in distances:
        atoms = to_ase(film_at(label, float(d)))
        atoms.calc = calculator
        energies.append(float(atoms.get_potential_energy()))
    energies = np.array(energies)
    minima = [refine_minimum(distances, energies, i)
              for i in range(1, len(energies) - 1)
              if energies[i] < energies[i - 1] and energies[i] < energies[i + 1]]
    chem = min((m for m in minima if m[0] < CHEMISORBED_BELOW), key=lambda m: m[1], default=None)
    phys = min((m for m in minima if m[0] >= CHEMISORBED_BELOW), key=lambda m: m[1], default=None)
    start = chem or phys
    if start is None:
        # A monotonic curve has no minimum to relax from — the expected outcome for the
        # dispersion-bound hollow registry when D3 is unavailable.
        scan_results[label] = {"distances": distances, "energies": energies,
                               "chem": None, "phys": None, "relaxed": None}
        print(f"{label:<10} unbound in this window — no minimum to relax from"
              + ("" if dispersion_active else " (dispersion inactive)"))
        continue
    relaxed_atoms = relax(to_ase(film_at(label, start[0])))
    separation, buckling, registry_now = interface_geometry(relaxed_atoms)
    expected_sites = {"atop_fcc": frozenset(("atop", "fcc")), "atop_hcp": frozenset(("atop", "hcp")),
                      "hollow": frozenset(("fcc", "hcp"))}.get(label)
    if expected_sites is not None and registry_now != expected_sites and None not in registry_now:
        print(f"! {label}: relaxed into {set(registry_now)} — treat its row with suspicion")
    energy = float(relaxed_atoms.get_potential_energy())
    scan_results[label] = {
        "distances": distances, "energies": energies, "chem": chem, "phys": phys,
        "relaxed": {"energy": energy, "separation": separation, "buckling": buckling,
                    "w_adh": (E_slab + E_sheet - energy) / area * EV_PER_A2_TO_J_PER_M2,
                    "material": Material.create(from_ase(relaxed_atoms))},
    }
    print(f"{label:<10} relaxed: d = {separation:5.2f} A   buckling = {buckling:+.3f} A   "
          f"W_adh = {scan_results[label]['relaxed']['w_adh']:.2f} J/m^2")


In [ ]:
import plotly.graph_objects as go

reference = min(min(m[1] for m in (r["chem"], r["phys"]) if m) for r in scan_results.values())
fig = go.Figure()
for label, r in scan_results.items():
    fig.add_trace(go.Scatter(x=r["distances"], y=(r["energies"] - reference) * 1000 / n_carbon,
                             mode="lines+markers", name=label))
fig.update_layout(
    title="Rigid-scan energy vs. separation (MACE-MP + D3) — bracketing only; the table below is relaxed",
    xaxis_title="plane distance (A)",
    yaxis_title="energy relative to the deepest minimum (meV / C atom)",
    yaxis_range=[-20, 300],
)
fig.show()


In [ ]:
# Lahiri et al. (2011), Table 1 — the published targets (the review quotes the hollow as 0.38)
PAPER = {
    "atop_fcc": {"w_adh": 0.81, "separation": 2.16},
    "atop_hcp": {"w_adh": 0.77, "separation": 2.17},
    "hollow": {"w_adh": 0.31, "separation": 3.26},
}
PAPER_BUCKLING = 0.03  # A, computed (the review, from ref. 35); LEED I-V measures 0.05 A
TOL_W = 0.15  # J/m^2
TOL_D = 0.10  # A

relaxed_rows = {k: v["relaxed"] for k, v in scan_results.items() if v["relaxed"] is not None}
print(f"{'registry':<10}{'W_adh J/m^2':<14}{'paper':<8}{'d (A)':<8}{'paper':<8}{'buckling (A)'}")
for label, r in sorted(relaxed_rows.items(), key=lambda kv: -kv[1]["w_adh"]):
    t = PAPER.get(label, {})
    print(f"{label:<10}{r['w_adh']:<14.2f}{t.get('w_adh', '—'):<8}"
          f"{r['separation']:<8.2f}{t.get('separation', '—'):<8}{r['buckling']:+.3f}")
for label, v in scan_results.items():
    if v["relaxed"] is None:
        print(f"{label:<10}unbound in this environment — paper: "
              f"{PAPER.get(label, {}).get('w_adh', '—')} J/m^2 at {PAPER.get(label, {}).get('separation', '—')} A")

def within(label, key, target, tol):
    row = relaxed_rows.get(label)
    return row is not None and abs(row[key] - target) <= tol

checks_mace = {
    "ordering fcc > hcp > hollow (W_adh)": (
        all(k in relaxed_rows for k in PAPER)
        and relaxed_rows["atop_fcc"]["w_adh"] > relaxed_rows["atop_hcp"]["w_adh"] > relaxed_rows["hollow"]["w_adh"]),
    "fcc W_adh within 0.15 J/m^2 of 0.81": within("atop_fcc", "w_adh", 0.81, TOL_W),
    "fcc separation within 0.10 A of 2.16": within("atop_fcc", "separation", 2.16, TOL_D),
    "hollow separation within 0.10 A of 3.26": within("hollow", "separation", 3.26, TOL_D),
    "atop carbon buckles outward": "atop_fcc" in relaxed_rows and relaxed_rows["atop_fcc"]["buckling"] > 0,
}
for name, ok in checks_mace.items():
    print(f"  {'ok  ' if ok else 'FAIL'} {name}")
print(f"\nReproduces Lahiri et al. Table 1 [MACE tier]: {'yes' if all(checks_mace.values()) else 'no'}")
reason = ("dispersion is inactive here, so this is the GGA-level picture the manuscript rejects"
          if not dispersion_active else
          "MACE-MP is PBE-trained, and PBE is the functional the manuscript rejects for this interface")
print(f"({reason} — the DFT tier below runs the paper's LDA and carries the reproduction claim)")


## 5. Precise Tier: the Paper's LDA, Relaxed, on the Platform

One relaxation + total-energy job per selected registry, starting from the MACE-relaxed geometry,
plus the two same-cell references the work of adhesion needs — the paper's functional (LDA),
spin-polarized, no dispersion correction. A default run selects one registry (three jobs). An
**empty** list skips the platform tier entirely, which is what the automated test does: with
relaxation these jobs take longer than a browser test may wait.


In [ ]:
DFT_REGISTRY_NAMES = [
    "atop_fcc",
    # "atop_hcp",
    # "hollow",
    # "bridge",
]


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"Using project: {projects[0]['name']} ({project_id})")

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

def submitted_copy(material, name):
    """QE needs ATOMIC_SPECIES and ATOMIC_POSITIONS to agree, and the film/substrate labels only
    served the displacement, so they are dropped from anything submitted."""
    m = material.clone()
    m.basis.labels.values = []
    m.name = name
    return Material.create(get_or_create_material(client, m, ACCOUNT_ID))

dft_materials, reference_materials = {}, {}
if DFT_REGISTRY_NAMES:
    for label in DFT_REGISTRY_NAMES:
        relaxed = scan_results[label]["relaxed"]
        saved = submitted_copy(relaxed["material"],
                               f"{BASE_MATERIAL_NAME} {label} d{relaxed['separation']:.2f} relaxed")
        dft_materials[label] = saved
        print(f"{label:<16} -> '{saved.name}' ({len(saved.basis.elements.values)} atoms)")
    # The references live in the same cell and run with the same settings, so the cell- and
    # sampling-dependent part of the error drops out of the work-of-adhesion difference.
    for name, part in (("substrate", substrate_part), ("film", film_part)):
        saved = submitted_copy(part, f"{BASE_MATERIAL_NAME} {name} reference")
        reference_materials[name] = saved
        print(f"{name + ' ref':<16} -> '{saved.name}' ({len(saved.basis.elements.values)} atoms)")
else:
    print("DFT tier skipped: no registries selected.")


In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

In [ ]:
from mat3ra.mode import ModelFactory
from mat3ra.standata.model_tree import ModelTreeStandata

# The paper's functional. LDA describes this interface's geometry in agreement with experiment,
# which is the stated reason Lahiri et al. chose it over GGA; no dispersion correction is added
# on top, matching the paper.
model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft",
    subtype=MODEL_SUBTYPE,
    functional=FUNCTIONAL,
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)

for subworkflow in workflow.subworkflows:
    subworkflow.model = model

# Relaxation is the point: the buckling is one of the published numbers, and a single point at the
# MACE geometry would inherit MACE's PBE-grade structure.
workflow.add_relaxation()


In [ ]:
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input

QE_UNIT_NAMES = ["pw_relax", "pw_scf"]

def system_patch_for(material):
    """&SYSTEM settings for one material. starting_magnetization is indexed by position in
    ATOMIC_SPECIES, so the index is looked up per material — a free-standing graphene reference
    contains no Ni and must not inherit its moment."""
    species_names = []
    for element in material.basis.elements.values:
        if element not in species_names:
            species_names.append(element)
    patch = {"nspin": 2, "degauss": DEGAUSS, "smearing": SMEARING}
    for atomic_species, value in STARTING_MAGNETIZATION.items():
        for index, name in enumerate(species_names):
            if name == atomic_species:
                patch[f"starting_magnetization({index + 1})"] = value
    return species_names, patch

def apply_calculation_settings(built, material):
    for unit_name in QE_UNIT_NAMES:
        for subworkflow in built.subworkflows:
            unit = subworkflow.get_unit_by_name(name=unit_name)
            if unit:
                unit.add_context(PointsGridDataProvider(material=material, dimensions=SCF_KGRID,
                                                        isEdited=True).get_context_item_data())
                unit.add_context(PlanewaveCutoffsContextProvider(wavefunction=ECUTWFC, density=ECUTRHO,
                                                                 isEdited=True).get_context_item_data())
                subworkflow.set_unit(unit)
    _, patch = system_patch_for(material)
    patch_workflow_qe_input(built, {"system": patch}, unit_names=QE_UNIT_NAMES)
    if ADDITIONAL_PARAMETERS:
        patch_workflow_qe_input(built, ADDITIONAL_PARAMETERS, unit_names=QE_UNIT_NAMES)
    return built

if dft_materials:
    reference_material = dft_materials[DFT_REGISTRY_NAMES[0]]
    apply_calculation_settings(workflow, reference_material)
    species_names, system_patch = system_patch_for(reference_material)
    print(f"ATOMIC_SPECIES order: {species_names}")
    print(f"&SYSTEM patch: {system_patch}")


In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflows = {}
if dft_materials:
    def configured_workflow(material, name):
        built = Workflow.create(WorkflowStandata.filter_by_application(app.name)
                                .get_by_name_first_match(WORKFLOW_SEARCH_TERM))
        built.name = name
        for subworkflow in built.subworkflows:
            subworkflow.model = model
        built.add_relaxation()
        return apply_calculation_settings(built, material)

    # One workflow per distinct element set, so a reference never inherits another material's
    # magnetization indices.
    workflows = {"interface": workflow}
    for name, material in reference_materials.items():
        if set(material.basis.elements.values) != set(reference_material.basis.elements.values):
            workflows[name] = configured_workflow(material, f"{MY_WORKFLOW_NAME} {name}")
        else:
            workflows[name] = workflow

    seen = {}
    for key, wf in workflows.items():
        if id(wf) not in seen:
            seen[id(wf)] = Workflow.create(get_or_create_workflow(client, wf, ACCOUNT_ID))
        saved_workflows[key] = seen[id(wf)]
        print(f"{key:<12} -> workflow {saved_workflows[key].id}")


In [ ]:
clusters = client.clusters.list() if dft_materials else []
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

In [ ]:
from mat3ra.ide.compute import Compute

compute = None
if dft_materials:
    if CLUSTER_NAME:
        cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    else:
        cluster = clusters[0]
    compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
    print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


In [ ]:
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.job import create_job

def submit_job_for(label, saved_material, which="interface"):
    job_response = create_job(
        api_client=client,
        materials=[saved_material],
        workflow=workflows[which],
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{MY_WORKFLOW_NAME} {label} {timestamp}",
        compute=compute.to_dict(),
    )
    job_id = dict_to_namespace_recursive(job_response)._id
    print(f"{label:<16} -> job {job_id}")
    return job_id

jobs, reference_jobs = {}, {}
if dft_materials:
    jobs = {label: submit_job_for(label, m) for label, m in dft_materials.items()}
    reference_jobs = {name: submit_job_for(f"{name} reference", m, which=name)
                      for name, m in reference_materials.items()}


In [ ]:
for label, job_id in {**jobs, **reference_jobs}.items():
    client.jobs.submit(job_id)
    print(f"Submitted {label}: {job_id}")


In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

all_job_ids = list(jobs.values()) + list(reference_jobs.values())
if all_job_ids:
    await wait_for_jobs_to_finish_async(client.jobs, all_job_ids, poll_interval=POLL_INTERVAL)
else:
    print("Nothing to wait for — the DFT tier was skipped.")


In [ ]:
from mat3ra.prode import PropertyName

dft_energies, reference_energies, dft_w_adh = {}, {}, {}
if jobs:
    def total_energy_of(job_id):
        property_data = client.properties.get_for_job(job_id, property_name=PropertyName.scalar.total_energy.value)
        return float(property_data[0]["data"]["value"])

    dft_energies = {label: total_energy_of(job_id) for label, job_id in jobs.items()}
    reference_energies = {name: total_energy_of(job_id) for name, job_id in reference_jobs.items()}
    separated = reference_energies["substrate"] + reference_energies["film"]
    dft_w_adh = {label: (separated - e) / area * EV_PER_A2_TO_J_PER_M2 for label, e in dft_energies.items()}

    print(f"{'registry':<12}{'E_DFT (eV)':<16}{'W_adh (J/m^2)':<15}{'paper (J/m^2)'}")
    for label, e in sorted(dft_energies.items(), key=lambda kv: kv[1]):
        t = PAPER.get(label, {})
        print(f"{label:<12}{e:<16.4f}{dft_w_adh[label]:<15.2f}{t.get('w_adh', '—')}")


## 6. Compare with the Article


In [ ]:
# The verdict, per tier, against Lahiri et al. (2011) Table 1 — reached through the review.
print("Targets: fcc 0.81 J/m^2 @ 2.16 A · hcp 0.77 @ 2.17 · hollow 0.31 @ 3.26 · "
      f"buckling ~{PAPER_BUCKLING} A, atop carbon out\n")

print(f"Reproduces Lahiri et al. Table 1 [MACE tier]: {'yes' if all(checks_mace.values()) else 'no'}"
      f"  ({sum(checks_mace.values())}/{len(checks_mace)} checks)")

if dft_w_adh:
    evaluated = {label: dft_w_adh[label] for label in PAPER if label in dft_w_adh}
    checks_dft = {f"{label} W_adh within {TOL_W} J/m^2 of {PAPER[label]['w_adh']}":
                  abs(w - PAPER[label]["w_adh"]) <= TOL_W for label, w in evaluated.items()}
    if len(evaluated) == len(PAPER):
        checks_dft["ordering fcc > hcp > hollow"] = (
            dft_w_adh["atop_fcc"] > dft_w_adh["atop_hcp"] > dft_w_adh["hollow"])
    for name, ok in checks_dft.items():
        print(f"  {'ok  ' if ok else 'FAIL'} {name}")
    partial = "" if len(evaluated) == len(PAPER) else f" ({len(evaluated)} of {len(PAPER)} registries)"
    print(f"Reproduces Lahiri et al. Table 1 [DFT tier]: "
          f"{'yes' if checks_dft and all(checks_dft.values()) else 'no'}{partial}")
else:
    print("DFT tier: not run — select registries in DFT_REGISTRY_NAMES for the paper's-functional verdict.")


## References

[1] Arjun Dahal, Matthias Batzill, "Graphene-nickel interfaces: a review",
Nanoscale 6(5), 2548 (2014). [DOI: 10.1039/c3nr05279f](https://doi.org/10.1039/c3nr05279f)

[2] Jayeeta Lahiri, Travis S. Miller, Andrew J. Ross, Lyudmyla Adamska, Ivan I. Oleynik,
Matthias Batzill, "Graphene growth and stability at nickel surfaces", New J. Phys. 13, 025001
(2011). [DOI: 10.1088/1367-2630/13/2/025001](https://doi.org/10.1088/1367-2630/13/2/025001)

[3] mat3ra-made: https://github.com/Exabyte-io/made

[4] MACE-MP-0 foundation models: https://github.com/ACEsuit/mace
